# 02 · The quantum feature extractor

A *quanvolutional* layer is the quantum analogue of a convolution: slide a small
window over the image, and at each position transform the patch and record some
numbers. The difference is that the transformation is a quantum circuit —
encode the patch into a quantum state, entangle, and measure.

This notebook builds the layer, verifies it against Qiskit's simulator, makes it
about 10,000× faster, and then asks the question the project actually cares
about: **does it beat a classical convolution?**

The answer is no. Working out precisely *why* not turns out to be the useful
part.

## Setup

In [11]:
# Environment setup — works on Colab and on a local checkout.
#
# Safe to run in any order and any number of times. If another notebook in this
# session already cloned the repo, this reuses it rather than trying to clone
# again (which fails with "destination path already exists", exit 128).

import os, sys, subprocess, pathlib, importlib

REPO_URL = "https://github.com/Pushkar0997/qi26_25.git"
REPO_DIR = "qi26_25"
IN_COLAB = "google.colab" in sys.modules

# A file that exists only in this repo, used to recognise a valid checkout.
MARKER = pathlib.Path("integration") / "features.py"


def _is_root(p):
    return (p / MARKER).exists()


def _find_root():
    """Locate the repo, checking three places in order.

    Searching UPWARD alone is not enough. On Colab every notebook gets a fresh
    kernel starting at /content, so a second notebook would look at /content and
    its parents, miss the /content/qi26_25 that the first notebook cloned, and
    try to clone again into a directory that already exists.
    """
    here = pathlib.Path.cwd().resolve()

    # 1. Here or above — a local checkout, or a notebook opened from notebooks/.
    for cand in [here, *here.parents]:
        if _is_root(cand):
            return cand

    # 2. One level down — a clone made earlier in this session.
    for cand in [here / REPO_DIR, pathlib.Path("/content") / REPO_DIR]:
        if _is_root(cand):
            return cand.resolve()

    return None


root = _find_root()
if root is None:
    target = pathlib.Path.cwd() / REPO_DIR
    if target.exists():
        raise SystemExit(
            f"{target} exists but does not contain {MARKER}.\n"
            f"It may be a partial clone. Remove it and re-run this cell:\n"
            f"    !rm -rf {target}")
    print("cloning repository ...")
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(target)], check=True)
    root = _find_root()
    if root is None:
        raise SystemExit("clone succeeded but the repo layout was not found")
else:
    print("using existing checkout")

os.chdir(root)

# Put the package directories on the path. These modules import each other by
# bare name, so the directories themselves go on sys.path, not the repo root.
for sub in ["integration", "track_b_search/oracle", "data", "benchmarking",
            "track_a_vision/quanvolutional", "track_a_vision/encoding"]:
    d = str(root / sub)
    if d not in sys.path:
        sys.path.insert(0, d)

# Install only what is missing — Colab already ships numpy/scipy/sklearn/PIL.
need = []
for mod, pkg in [("qiskit", "qiskit>=2.0,<3.0"), ("qiskit_aer", "qiskit-aer"),
                 ("sklearn", "scikit-learn"), ("scipy", "scipy"),
                 ("PIL", "pillow"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
if need:
    print("installing:", " ".join(need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import qiskit, numpy as np
print("repo root :", root)
print("colab     :", IN_COLAB)
print("qiskit    :", qiskit.__version__, "| numpy:", np.__version__)

using existing checkout
repo root : /content/qi26_25
colab     : True
qiskit    : 2.5.2 | numpy: 2.0.2


In [12]:
# Generate the dataset if it is not already present (~30 s, deterministic).
#
# IMPORTANT: check for an actual data file, not the manifest. manifest.json is
# committed to the repo (it records the seed, fonts and platform used, which the
# report cites), but the ~8 MB of images and .npz files are gitignored because
# they regenerate exactly from the seed. So on a fresh clone the manifest EXISTS
# while the data does not, and testing the manifest alone wrongly concludes
# everything is ready.
from pathlib import Path
import subprocess, sys, json

TIERS = ["clean_digital", "clean_scan", "noisy_scan",
         "clear_handwriting", "degraded_handwriting"]
have_data = all((Path("data/processed") / t / "chars.npz").exists() for t in TIERS)

if have_data:
    print("dataset already present")
else:
    print("generating dataset (about 30 s) ...")
    subprocess.run([sys.executable, "data/generate_dataset.py",
                    "--docs-per-tier", "12", "--seed", "26"], check=True)

m = json.loads(Path("data/processed/manifest.json").read_text())
print()
print("seed        :", m["seed"])
print("fonts       :", Path(m["font_print"]).name, "/", Path(m["font_hand"]).name)
print("platform    :", m["platform"])
total = sum(t["n_chars"] for t in m["tiers"].values())
for name, t in m["tiers"].items():
    print("  {:22s} {:2d} docs  {:5d} crops".format(name, t["n_docs"], t["n_chars"]))
print("total crops :", total)

dataset already present

seed        : 26
fonts       : LiberationSans-Regular.ttf / LiberationSerif-Italic.ttf
platform    : linux
  clean_digital          12 docs   1207 crops
  clean_scan             12 docs   1217 crops
  noisy_scan             12 docs   1200 crops
  clear_handwriting      12 docs   1193 crops
  degraded_handwriting   12 docs   1196 crops
total crops : 6013


## 1 · The filter circuit

Four qubits, one per pixel of a 2×2 patch. Two layers of RY rotations with a
ring of CX gates between them. The rotation angles are drawn once from a fixed
seed and then held constant — this is the *untrained random* filter from
Henderson et al.'s quanvolutional networks. Notebook `04` and the report cover
what happens when you train them instead.

In [13]:
from features import random_entangling_circuit

filt = random_entangling_circuit(n_qubits=4, seed=42, depth=2)
print(filt.draw("text"))
print("\nDepth:", filt.depth(), "| gates:", dict(filt.count_ops()))

     ┌────────────┐                      ┌───┐┌─────────────┐               »
q_0: ┤ Ry(4.8629) ├──■───────────────────┤ X ├┤ Ry(0.59173) ├──■────────────»
     ├────────────┤┌─┴─┐     ┌──────────┐└─┬─┘└─────────────┘┌─┴─┐          »
q_1: ┤ Ry(2.7576) ├┤ X ├──■──┤ Ry(6.13) ├──┼─────────────────┤ X ├──■───────»
     ├────────────┤└───┘┌─┴─┐└──────────┘  │   ┌────────────┐└───┘┌─┴─┐     »
q_2: ┤ Ry(5.3947) ├─────┤ X ├─────■────────┼───┤ Ry(4.7824) ├─────┤ X ├──■──»
     ├────────────┤     └───┘   ┌─┴─┐      │   ├───────────┬┘     └───┘┌─┴─┐»
q_3: ┤ Ry(4.3817) ├─────────────┤ X ├──────■───┤ Ry(4.939) ├───────────┤ X ├»
     └────────────┘             └───┘          └───────────┘           └───┘»
«     ┌───┐
«q_0: ┤ X ├
«     └─┬─┘
«q_1: ──┼──
«       │  
«q_2: ──┼──
«       │  
«q_3: ──■──
«          

Depth: 10 | gates: {'ry': 8, 'cx': 8}


## 2 · Encoding a patch

Each pixel value $v \in [0, 255]$ becomes a rotation angle $\theta = \pi v/255$
applied to its own qubit. A qubit in state $R_Y(\theta)|0\rangle$ is
$\cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$, so a dark pixel stays near
|0⟩ and a bright one rotates toward |1⟩.

Because each qubit is rotated independently, the four-qubit input state is a
**product state** — a Kronecker product of four single-qubit states, with no
entanglement. All the entanglement in the output comes from the filter's CX
gates. That fact is what makes section 4 possible.

In [14]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

patch = np.array([[10, 240], [250, 15]])       # a diagonal edge
print("patch:\n", patch)

qc = QuantumCircuit(4)
for i, v in enumerate(patch.flatten()):
    qc.ry(np.pi * v / 255, i)

sv = Statevector(qc).data
print("\ninput statevector (product state, 16 amplitudes):")
print(np.round(np.abs(sv), 3))
print("\nlargest amplitude at basis state |{:04b}> — "
      "the two bright pixels dominate".format(int(np.argmax(np.abs(sv)))))

patch:
 [[ 10 240]
 [250  15]]

input statevector (product state, 16 amplitudes):
[0.003 0.    0.03  0.002 0.092 0.006 0.989 0.061 0.    0.    0.003 0.
 0.008 0.001 0.092 0.006]

largest amplitude at basis state |0110> — the two bright pixels dominate


## 3 · Running it the obvious way (and why that will not scale)

Compose encoding with the filter, measure, count. This is the textbook approach
and it is correct — it is just slow.

In [15]:
from qiskit import transpile
from qiskit_aer import AerSimulator
import time

def quanv_patch_aer(patch, filt, shots=20000):
    qc = QuantumCircuit(4)
    for i, v in enumerate(patch.flatten().astype(float)):
        qc.ry(np.pi * v / 255, i)
    qc.compose(filt, inplace=True)
    qc.measure_all()
    counts = AerSimulator().run(transpile(qc, AerSimulator()),
                                shots=shots).result().get_counts()
    p1 = np.zeros(4)
    for bits, n in counts.items():
        b = bits.replace(" ", "")[::-1]
        for i in range(4):
            if b[i] == "1":
                p1[i] += n
    return p1 / shots

t0 = time.time()
probs_aer = quanv_patch_aer(patch, filt)
dt = time.time() - t0
print("P(qubit measures 1):", np.round(probs_aer, 4))
print(f"time for ONE patch: {dt:.2f} s")
print(f"\nAt 16 patches per character and ~6000 characters, that is "
      f"{16*6000*dt/3600:.1f} hours for one pass over the dataset.")

P(qubit measures 1): [0.5664 0.4547 0.5434 0.5964]
time for ONE patch: 0.18 s

At 16 patches per character and ~6000 characters, that is 4.9 hours for one pass over the dataset.


## 4 · The same computation, ~10,000× faster

The filter is a **fixed** circuit, so it is a single 16×16 unitary matrix $U$ —
the same for every patch. And the encoded input is a product state, so it can be
built with Kronecker products directly.

So the whole layer is $|\psi_{out}\rangle = U\,|\psi_{in}\rangle$: one
matrix–vector product per patch, and the entire dataset can go through as one
batched matmul.

**This is not an approximation.** It is the same linear algebra Aer performs,
minus per-job overhead and minus shot noise. Verify that claim rather than
trusting it.

In [16]:
from features import quanv_patch_probs, verify_against_qiskit

fast = quanv_patch_probs(patch[None, ...])[0]
print("Aer  (20k shots):", np.round(probs_aer, 4))
print("fast (exact)    :", np.round(fast, 4))
print("difference      :", np.round(np.abs(fast - probs_aer), 4))
print("\n-> agreement is at the level of sampling error, as it should be.\n")

err, ok = verify_against_qiskit(n_trials=6, shots=200000)
print(f"formal check over 6 random patches at 200k shots:")
print(f"  max deviation {err:.4f} -> {'MATCH' if ok else 'MISMATCH'}")

Aer  (20k shots): [0.5664 0.4547 0.5434 0.5964]
fast (exact)    : [0.5615 0.4573 0.5381 0.6038]
difference      : [0.0049 0.0026 0.0053 0.0074]

-> agreement is at the level of sampling error, as it should be.

formal check over 6 random patches at 200k shots:
  max deviation 0.0028 -> MATCH


In [17]:
import time
imgs = np.random.default_rng(0).integers(0, 256, (2000, 8, 8))
t0 = time.time()
from features import quanv_features
F = quanv_features(imgs)
print(f"2000 characters -> {F.shape[1]} features each in {time.time()-t0:.2f} s")
print("(the Aer path would take hours for the same work)")

2000 characters -> 64 features each in 0.05 s
(the Aer path would take hours for the same work)


## 5 · What you measure matters more than what circuit you ran

Here is the first genuinely surprising result.

The obvious readout is the per-qubit probability of measuring 1 — four numbers.
But a 4-qubit circuit produces a distribution over **16** outcomes, and keeping
only 4 marginals throws most of it away. In particular it discards every
*correlation* the entangling layer created, which is precisely the part a
classical product-state model could not have produced.

Adding pairwise correlations $\langle Z_i Z_j \rangle$ recovers six more
numbers per patch.

In [18]:
p_marg = quanv_patch_probs(patch[None, ...], correlations=False)[0]
p_full = quanv_patch_probs(patch[None, ...], correlations=True)[0]

print("marginals only  (4 values):", np.round(p_marg, 3))
print("with <Z_i Z_j> (10 values):", np.round(p_full, 3))
print("\nThe last 6 are the pairwise correlations, ranging over [-1, +1].")
print("A value near 0 means those qubits are uncorrelated; +-1 means locked together.")

marginals only  (4 values): [0.562 0.457 0.538 0.604]
with <Z_i Z_j> (10 values): [ 0.562  0.457  0.538  0.604 -0.492  0.588 -0.23  -0.812  0.408 -0.496]

The last 6 are the pairwise correlations, ranging over [-1, +1].
A value near 0 means those qubits are uncorrelated; +-1 means locked together.


## 6 · The central comparison

Now the question the project exists to answer. Four feature extractors, the same
crops, the same classifier, the same split:

| extractor | what it is |
|---|---|
| `quanv (marginals)` | quantum filter, 4 values per patch → 64 features |
| `quanv (+ZZ)` | quantum filter, 10 values per patch → 160 features |
| `classical conv` | random classical filters, **matched to 160 features** |
| `raw pixels` | no feature extraction at all, just the 64 pixel values |

Matching dimensionality is the point. Without it, a win is indistinguishable from
simply handing one method a wider representation to work with.

This takes about a minute.

In [19]:
from features import classical_conv, raw_pixels, normalize_crops
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

TIERS = ["clean_digital", "clean_scan", "noisy_scan",
         "clear_handwriting", "degraded_handwriting"]
X = np.concatenate([np.load(f"data/processed/{t}/chars.npz")["crops"] for t in TIERS])
y = np.concatenate([np.load(f"data/processed/{t}/chars.npz")["labels"] for t in TIERS])
Xn = normalize_crops(X)
print("crops:", X.shape, "| classes:", len(np.unique(y)))

extractors = {
    "quanv (marginals)": quanv_features(Xn),
    "quanv (+ZZ corr)":  quanv_features(Xn, correlations=True),
    "classical conv":    classical_conv(Xn, n_filters=10),
    "raw pixels":        raw_pixels(Xn),
}

print(f"\n{'extractor':22s} {'dim':>5s} {'accuracy':>10s}")
scores = {}
for name, F in extractors.items():
    a, b, c, d = train_test_split(F, y, test_size=.25, random_state=26, stratify=y)
    acc = LogisticRegression(max_iter=3000, C=5.).fit(a, c).score(b, d)
    scores[name] = acc
    print(f"{name:22s} {F.shape[1]:5d} {acc:10.3f}")

crops: (6013, 8, 8) | classes: 36

extractor                dim   accuracy
quanv (marginals)         64      0.870
quanv (+ZZ corr)         160      0.924
classical conv           160      0.936
raw pixels                64      0.942


### Reading this honestly

Three findings, in order of how much they matter:

**1. At matched dimensionality, the quantum layer ties its classical analogue.**
Not worse *specifically* — a random classical convolution with the same patch
size, stride, and output width scores the same within noise. Whatever is
limiting performance is not quantumness.

**2. Both lose to raw pixels — at lower dimensionality.** This is the more
informative result and it reframes the whole comparison. The relevant axis is not
quantum versus classical, it is *untrained random feature map* versus *no feature
map*. At 8×8 the input is already close to information-minimal, so any fixed
random projection discards more than it contributes. The classical control fails
in the same way and to the same degree, which is what identifies the cause.

**3. Readout design beat filter design.** Switching from marginals to
marginals-plus-correlations gained several points — a far larger effect than any
difference between the quantum and classical filters.

The obvious next question is whether *training* the filter changes this. The
brief asked for a variational circuit, so that experiment exists: see
`track_a_vision/quanvolutional/train_filter.py` and §7 of the report. Short
version — it does not help, and the leakage-free way of measuring that is itself
instructive.

## 7 · Where the quantum layer does hold an advantage

One thing does survive, and it is worth stating precisely because it is easy to
overstate.

The quantum filter produces its 160-dimensional feature space from **8
parameters** (the RY angles). The dimension-matched classical control needs
**40** (ten filters × four weights). That is a 5× parameter economy at equal
representational width, and at the untrained baseline the two perform the same —
so the economy is real and costs nothing there.

It does not survive training under the protocol tested, and the report says so.
But as an architectural property — a small number of parameters generating a
wide feature space through entanglement — it is the genuine quantum-side result
of this track.

In [20]:
print(f"quantum filter    : 8 parameters -> {extractors['quanv (+ZZ corr)'].shape[1]} features")
print(f"classical control : 40 parameters -> {extractors['classical conv'].shape[1]} features")
print(f"\nparameter economy : {40/8:.0f}x, at equal feature dimensionality")
print(f"accuracy at that point: quantum {scores['quanv (+ZZ corr)']:.3f} "
      f"vs classical {scores['classical conv']:.3f}")

quantum filter    : 8 parameters -> 160 features
classical control : 40 parameters -> 160 features

parameter economy : 5x, at equal feature dimensionality
accuracy at that point: quantum 0.924 vs classical 0.936
